In [1]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

load_dotenv()

project = os.getenv("GOOGLE_CLOUD_PROJECT")

# Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    vertexai=True,
    project=project
)

In [6]:
from langchain.tools import tool

@tool
def say_hello(name:str)->str:
    """This function is used to greet a person

    Args:
      name (str): name

    returns:
      str: Greeting

   """

    return f"Hello {name}"

In [7]:
llm_with_tools = llm.bind_tools([say_hello])

In [18]:
#llm which is not aware of tools
response = llm.invoke("Greet Anudeep")
response.pretty_print()
response

================================== Ai Message ==================================

Hello Anudeep! How are you doing today?


AIMessage(content='Hello Anudeep! How are you doing today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a09062-c474-7f03-b0a3-d3fcf69113a2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4, 'output_tokens': 11, 'total_tokens': 15, 'input_token_details': {'cache_read': 0}})

In [17]:
#llm with tools response
response = llm_with_tools.invoke("Greet Anudeep")
response.pretty_print()
response

================================== Ai Message ==================================
Tool Calls:
  say_hello (13c23cad-f541-4b9b-9e66-083eb0f59b91)
 Call ID: 13c23cad-f541-4b9b-9e66-083eb0f59b91
  Args:
    name: Anudeep


AIMessage(content='', additional_kwargs={'function_call': {'name': 'say_hello', 'arguments': '{"name": "Anudeep"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a09062-a4b0-7350-8ed3-e3594d79d1c5-0', tool_calls=[{'name': 'say_hello', 'args': {'name': 'Anudeep'}, 'id': '13c23cad-f541-4b9b-9e66-083eb0f59b91', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 7, 'total_tokens': 46, 'input_token_details': {'cache_read': 0}})

In [20]:
from langchain.agents import create_agent

#create an agent 

agent = create_agent(
    model =llm,
    tools=[say_hello]
)

In [28]:
agent_response = agent.invoke({
    "messages": [
        ("human", "Greet shyam")
    ]
})

In [30]:
#agent_response['messages'][-1].pretty_print()
agent_response['messages']

[HumanMessage(content='Greet shyam', additional_kwargs={}, response_metadata={}, id='d02e872a-25bc-4405-8543-64d141bc7227'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'say_hello', 'arguments': '{"name": "shyam"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a090e8-99a7-71f1-8a03-20c19a6ce7cd-0', tool_calls=[{'name': 'say_hello', 'args': {'name': 'shyam'}, 'id': '4ae4e9d6-4042-429b-9690-56eef25e2293', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 6, 'total_tokens': 44, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='Hello shyam', name='say_hello', id='4cdd9160-00f8-424a-bdc2-6b7ba0b6b775', tool_call_id='4ae4e9d6-4042-429b-9690-56eef25e2293'),
 AIMessage(content='Hello shyam', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 's

In [38]:
!uv add langchain-tavily

Resolved 89 packages in 101ms
Checked 84 packages in 26ms


In [39]:
# I want to search internet
load_dotenv()
#os.getenv('TAVILY_API_KEY')

True

In [42]:
#lets create tavily search tool

from langchain_tavily import TavilySearch

tavily_search_tool = TavilySearch(
    max_results = 3,
    topic ="finance"
)

In [43]:
#lets create an agent with tavily search tool
agent = create_agent(
    model = llm,
    tools=[tavily_search_tool]
)

In [48]:
response = agent.invoke({
    "messages": "get the latest news on stocks"
})

In [49]:
response['messages']

[HumanMessage(content='get the latest news on stocks', additional_kwargs={}, response_metadata={}, id='8d1ac10c-e27a-4f02-b449-1e08b794fdfe'),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "stocks", "topic": "finance"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a093a9-ab5e-7112-a91f-f1d0d16d1a74-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'stocks', 'topic': 'finance'}, 'id': '2846b56e-5c27-42ac-b941-73b4d4a676c6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1295, 'output_tokens': 9, 'total_tokens': 1304, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='{"query": "stocks", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://ts2.tech/en/energy-stocks-outlook-2026-dec-25-2025-news-roundup-on-oil-prices-lng-natura